## [Swin Transformer](https://arxiv.org/abs/2103.14030)

The **Swin Transformer** (Shifted Window Transformer) is a hierarchical vision backbone that addresses two key limitations of the original ViT:

1. **ViT uses global attention** — every patch attends to every other patch. This is O(N²) in the number of patches, making it expensive for high-resolution images and impractical for dense prediction tasks (detection, segmentation).
2. **ViT produces a single-scale feature map** — no multi-scale hierarchy like ResNet, so it can't be plugged into standard detection/segmentation pipelines.

Swin solves both by computing **self-attention within local windows** and **shifting windows between layers** to allow cross-window connections.

<img src="./figures/swin_teaser.png" title="Shifted window partitioning" width="500"/>

At layer *l*, the image is partitioned into non-overlapping windows. At layer *l+1*, the partition is **shifted** by half a window size — so patches at window boundaries now share a window and can attend to each other.

### Swin Transformer Architecture

<img src="./figures/swin_arch.png" title="Swin Transformer architecture" width="900"/>

Swin has **four stages** with decreasing spatial resolution and increasing channels — exactly like ResNet:

| Stage | Resolution | Channels | Blocks |
|-------|-----------|----------|--------|
| 1 | H/4 × W/4 | C | 2 |
| 2 | H/8 × W/8 | 2C | 2 |
| 3 | H/16 × W/16 | 4C | 6 |
| 4 | H/32 × W/32 | 8C | 2 |

**Patch Partition**: Split image into 4×4 patches → tokens of dim 48 (4×4×3).  
**Linear Embedding**: Project 48-dim tokens to hidden dim C.  
**Patch Merging**: Between stages, merge 2×2 neighboring tokens → halve spatial size, double channels.  
**Swin Block**: Alternating W-MSA (window attention) and SW-MSA (shifted-window attention).

### Shifted Window Self-Attention

<img src="./figures/swin_shifted_window.png" title="Cyclic shift for shifted window attention" width="700"/>

Instead of a new partition, Swin uses a **cyclic shift** — roll the feature map by half a window size, compute masked attention within the shifted windows, then reverse the shift. This keeps the same number of windows (no extra cost) while enabling cross-window information flow.

**Complexity comparison** (H×W image, window size M×M):

$$\text{ViT global MSA: } \mathcal{O}(H^2 W^2 C) \qquad \text{Swin W-MSA: } \mathcal{O}(HWM^2C)$$

With M=7 (the default), W-MSA is linear in image size — global attention is quadratic.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

os.environ['http_proxy']  = 'http://192.41.170.23:3128'
os.environ['https_proxy'] = 'http://192.41.170.23:3128'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import math

from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader

from torchvision.datasets.mnist import MNIST
from torchvision.transforms import ToTensor, Compose, Resize

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device', device)

## Dataset and Dataloader

We resize MNIST to 32×32 so the spatial size is cleanly divisible into 4×4 patches across multiple stages (32/4 = 8, 8/2 = 4).

In [ ]:
from torch.utils.data import random_split

transform = Compose([Resize((32, 32)), ToTensor()])

full_dataset = MNIST(root='./data/', train=True, download=True, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])
test_set = MNIST(root='./data/', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, shuffle=True,  batch_size=64)
val_loader   = DataLoader(val_set,   shuffle=False, batch_size=64)
test_loader  = DataLoader(test_set,  shuffle=False, batch_size=64)

len(train_loader), len(val_loader), len(test_loader)

In [ ]:
for images, labels in train_loader:
    break

print(images.shape)   # (64, 1, 32, 32)
print(labels.shape)

the_image = images[0].permute(1, 2, 0)
plt.figure(figsize=(2, 2))
plt.imshow(the_image, cmap='gray')
plt.title(f'Label: {labels[0].item()}')
plt.axis('off')
plt.show()

## Swin Transformer Model

We build a small Swin Transformer for MNIST classification. The implementation follows the paper closely.

### Visualizing the Swin Pipeline: Image → Patches → Windows

Before diving into code, let's see what actually happens to an image as it flows through the first steps of Swin:

1. **Original image** (32×32 pixels)
2. **Patch partition** — split into 4×4 patches → 8×8 token grid (64 tokens)
3. **Window partition** — group the 8×8 grid into 4×4 windows → 4 windows (each window = 16 tokens that attend to each other)
4. **Shifted window** — same thing after cyclic shift by 2 → different groupings for cross-window connections

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Use the first image from the batch
img = images[0, 0].numpy()  # (32, 32)

PATCH_SIZE  = 4   # each patch is 4x4 pixels
WINDOW_SIZE = 4   # each window covers 4x4 patches
SHIFT_SIZE  = 2   # half of window_size
N_PATCHES   = 32 // PATCH_SIZE   # = 8 (token grid side)

# Colors for 4 windows (normal) and 9 regions (shifted)
WINDOW_COLORS  = ["#4E9AF1", "#F4A261", "#2EC4B6", "#E63946"]
SHIFTED_COLORS = ["#4E9AF1","#F4A261","#2EC4B6","#E63946",
                  "#9B5DE5","#F15BB5","#FEE440","#00BBF9","#00F5D4"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# ── Panel 1: Original image ──────────────────────────────────────────────────
axes[0].imshow(img, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("1. Original image
(32×32 pixels)", fontsize=11)
axes[0].axis("off")

# ── Panel 2: Patch partition ─────────────────────────────────────────────────
axes[1].imshow(img, cmap="gray", vmin=0, vmax=1)
for i in range(0, 32, PATCH_SIZE):
    axes[1].axhline(i - 0.5, color="red", linewidth=0.8)
    axes[1].axvline(i - 0.5, color="red", linewidth=0.8)
axes[1].set_title(f"2. Patch partition
(4×4 patches → {N_PATCHES}×{N_PATCHES} token grid)", fontsize=11)
axes[1].axis("off")

# ── Panel 3: Window partition (W-MSA) ────────────────────────────────────────
axes[2].imshow(img, cmap="gray", vmin=0, vmax=1, alpha=0.35)
win_px = WINDOW_SIZE * PATCH_SIZE   # window side in pixels = 4*4 = 16
win_idx = 0
for wy in range(0, 32, win_px):
    for wx in range(0, 32, win_px):
        color = WINDOW_COLORS[win_idx % len(WINDOW_COLORS)]
        rect  = mpatches.FancyBboxPatch(
            (wx - 0.5, wy - 0.5), win_px, win_px,
            boxstyle="square,pad=0", linewidth=2,
            edgecolor="white", facecolor=color, alpha=0.55
        )
        axes[2].add_patch(rect)
        axes[2].text(wx + win_px/2, wy + win_px/2,
                     f"W{win_idx}", ha="center", va="center",
                     fontsize=13, fontweight="bold", color="white")
        win_idx += 1
axes[2].set_xlim(-0.5, 31.5)
axes[2].set_ylim(31.5, -0.5)
axes[2].set_title(f"3. Window partition (W-MSA)
({N_PATCHES//WINDOW_SIZE}×{N_PATCHES//WINDOW_SIZE} = 4 windows, each {WINDOW_SIZE}×{WINDOW_SIZE} tokens)", fontsize=11)
axes[2].axis("off")

# ── Panel 4: Shifted window partition (SW-MSA) ───────────────────────────────
shift_px = SHIFT_SIZE * PATCH_SIZE   # shift in pixels = 2*4 = 8
axes[3].imshow(img, cmap="gray", vmin=0, vmax=1, alpha=0.35)

# Build a label map for the shifted grid (same logic as _build_mask)
label_map = np.full((32, 32), -1, dtype=int)
h_bounds = [0, 32 - win_px, 32 - shift_px, 32]
w_bounds = [0, 32 - win_px, 32 - shift_px, 32]
region = 0
for hi in range(len(h_bounds) - 1):
    for wi in range(len(w_bounds) - 1):
        label_map[h_bounds[hi]:h_bounds[hi+1], w_bounds[wi]:w_bounds[wi+1]] = region
        region += 1

# Draw each unique region as a colored block
for r in range(9):
    ys, xs = np.where(label_map == r)
    if len(ys) == 0:
        continue
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    color = SHIFTED_COLORS[r]
    rect = mpatches.FancyBboxPatch(
        (x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0,
        boxstyle="square,pad=0", linewidth=1.5,
        edgecolor="white", facecolor=color, alpha=0.55
    )
    axes[3].add_patch(rect)
    axes[3].text((x0 + x1) / 2, (y0 + y1) / 2,
                 f"R{r}", ha="center", va="center",
                 fontsize=9, fontweight="bold", color="white")

# Draw the 4 shifted window borders
for wy in range(0, 32, win_px):
    for wx in range(0, 32, win_px):
        rect = mpatches.FancyBboxPatch(
            ((wx + shift_px) % 32 - 0.5, (wy + shift_px) % 32 - 0.5),
            win_px, win_px,
            boxstyle="square,pad=0", linewidth=2,
            edgecolor="black", facecolor="none"
        )
        # Just draw the window borders via grid lines for clarity
for i in [shift_px, shift_px + win_px]:
    if i < 32:
        axes[3].axhline(i - 0.5, color="black", linewidth=1.5, linestyle="--")
        axes[3].axvline(i - 0.5, color="black", linewidth=1.5, linestyle="--")

axes[3].set_xlim(-0.5, 31.5)
axes[3].set_ylim(31.5, -0.5)
axes[3].set_title(f"4. Shifted window (SW-MSA)
(shift by {SHIFT_SIZE} tokens — regions cross window boundaries)", fontsize=11)
axes[3].axis("off")

plt.suptitle(f"Image → Patches → Windows  (patch={PATCH_SIZE}px, window={WINDOW_SIZE} patches)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Token grid : {N_PATCHES}×{N_PATCHES} = {N_PATCHES**2} tokens")
print(f"W-MSA      : {(N_PATCHES//WINDOW_SIZE)**2} windows × {WINDOW_SIZE**2} tokens each")
print(f"Global attn: would need to attend over all {N_PATCHES**2} tokens at once")

#### Step 1: Patch Partition and Linear Embedding

Split the input image into non-overlapping 4×4 patches, then project each patch to hidden dim C.

For a 32×32 image with 4×4 patches:
$$(N, 1, 32, 32) \rightarrow (N, 8\times8, 4\times4\times1) \rightarrow (N, 64, C)$$

We get an 8×8 grid of tokens (H/4 × W/4) for Stage 1.

In [ ]:
def patch_partition(x, patch_size=4):
    """(N, C, H, W) -> (N, H/P * W/P, P*P*C)"""
    N, C, H, W = x.shape
    P = patch_size
    x = x.reshape(N, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)   # (N, H/P, W/P, P, P, C)
    x = x.reshape(N, (H // P) * (W // P), P * P * C)
    return x

patches = patch_partition(images, patch_size=4)
print('Patches shape:', patches.shape)  # (64, 64, 16)

#### Step 2: Window Partition

Instead of global attention over all 64 tokens, we partition the 8×8 token grid into **local windows** of size M×M and compute attention only within each window.

For an 8×8 grid with M=4:
- Number of windows = (8/4)×(8/4) = **4 windows**
- Each window has 4×4 = **16 tokens**
- Attention is O(M²) per window, O(HW·M²) overall — linear in image size

In [ ]:
def window_partition(x, window_size):
    """(N, H, W, C) -> (num_windows*N, window_size, window_size, C)"""
    N, H, W, C = x.shape
    x = x.view(N, H // window_size, window_size, W // window_size, window_size, C)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(-1, window_size, window_size, C)

def window_reverse(windows, window_size, H, W):
    """Reverse window_partition back to (N, H, W, C)."""
    N = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(N, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(N, H, W, -1)

# Test
token_grid = patches.reshape(-1, 8, 8, 16)  # (N, 8, 8, 16)
windows = window_partition(token_grid, window_size=4)
print('Windows shape:', windows.shape)  # (N*4, 4, 4, 16)

#### Step 3: Window Multi-Head Self Attention (W-MSA)

Standard multi-head self-attention, but applied **per window** independently. Each window's M² tokens attend only to each other.

Swin also adds a **relative position bias** to the attention scores:
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}} + B\right)V$$

where $B$ is a learnable bias indexed by the relative position of each token pair within the window. This is more flexible than the fixed sinusoidal positional encoding in ViT.

In [ ]:
class WindowAttention(nn.Module):
    """Window multi-head self-attention with relative position bias."""

    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads
        self.scale       = (dim // num_heads) ** -0.5

        # Learnable relative position bias table: (2W-1)^2 entries per head
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) ** 2, num_heads)
        )
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        # Precompute relative position index for every token pair in a window
        coords   = torch.arange(window_size)
        grid     = torch.stack(torch.meshgrid(coords, coords, indexing='ij'))  # (2, W, W)
        flat     = grid.flatten(1)                                              # (2, W²)
        rel      = flat[:, :, None] - flat[:, None, :]                         # (2, W², W²)
        rel      = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += window_size - 1
        rel[:, :, 1] += window_size - 1
        rel[:, :, 0] *= 2 * window_size - 1
        self.register_buffer('relative_position_index', rel.sum(-1))  # (W², W²)

        self.qkv     = nn.Linear(dim, dim * 3, bias=True)
        self.proj    = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        # x: (num_windows*N, W², C)
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)  # each (B_, heads, N, head_dim)

        attn = (q @ k.transpose(-2, -1)) * self.scale

        # Add relative position bias
        bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        bias = bias.view(self.window_size ** 2, self.window_size ** 2, -1).permute(2, 0, 1).unsqueeze(0)
        attn = attn + bias

        # Apply attention mask for shifted windows
        if mask is not None:
            nW   = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        return self.proj((attn @ v).transpose(1, 2).reshape(B_, N, C))

#### Step 4: Swin Transformer Block (W-MSA / SW-MSA)

Each block follows the same LN → attention → residual → LN → MLP → residual pattern as ViT:

$$\hat{z}^l = \text{W-MSA}(\text{LN}(z^{l-1})) + z^{l-1}$$
$$z^l = \text{MLP}(\text{LN}(\hat{z}^l)) + \hat{z}^l$$

The `shift_size` parameter switches between W-MSA (`shift_size=0`) and SW-MSA (`shift_size=window_size//2`). Blocks come in pairs — one of each — so the receptive field grows with depth.

In [ ]:
class SwinTransformerBlock(nn.Module):

    def __init__(self, dim, input_resolution, num_heads, window_size=4, shift_size=0, mlp_ratio=4.0):
        super().__init__()
        self.input_resolution = input_resolution  # (H, W) in tokens
        self.window_size       = window_size
        self.shift_size        = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn  = WindowAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)

        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, dim),
        )

        # Precompute attention mask for SW-MSA to zero out cross-region pairs
        self.register_buffer('attn_mask', self._build_mask() if shift_size > 0 else None)

    def _build_mask(self):
        H, W = self.input_resolution
        img_mask = torch.zeros(1, H, W, 1)
        h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1
        mask_windows = window_partition(img_mask, self.window_size).view(-1, self.window_size ** 2)
        mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        return mask.masked_fill(mask != 0, -100.0).masked_fill(mask == 0, 0.0)

    def forward(self, x):
        H, W = self.input_resolution
        N, L, C = x.shape
        shortcut = x
        x = self.norm1(x).view(N, H, W, C)

        # Cyclic shift for SW-MSA
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))

        # Partition windows → attend → reverse
        x_win = window_partition(x, self.window_size).view(-1, self.window_size ** 2, C)
        x_win = self.attn(x_win, mask=self.attn_mask)
        x     = window_reverse(x_win.view(-1, self.window_size, self.window_size, C), self.window_size, H, W)

        # Reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))

        x = shortcut + x.view(N, L, C)
        x = x + self.mlp(self.norm2(x))
        return x

#### Step 5: Patch Merging (Downsampling Between Stages)

To build the hierarchical feature pyramid, **Patch Merging** is applied between stages:

1. Concatenate every 2×2 group of neighboring tokens: C → 4C
2. LayerNorm
3. Linear projection 4C → 2C

Result: spatial resolution halves, channels double — same effect as stride-2 convolution.

$$(N, H\times W, C) \rightarrow (N, \tfrac{H}{2}\times\tfrac{W}{2},\ 2C)$$

In [ ]:
class PatchMerging(nn.Module):

    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.norm      = nn.LayerNorm(4 * dim)
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        H, W = self.input_resolution
        N, L, C = x.shape
        x = x.view(N, H, W, C)

        x0 = x[:, 0::2, 0::2, :]   # top-left
        x1 = x[:, 1::2, 0::2, :]   # bottom-left
        x2 = x[:, 0::2, 1::2, :]   # top-right
        x3 = x[:, 1::2, 1::2, :]   # bottom-right

        x = torch.cat([x0, x1, x2, x3], dim=-1).view(N, -1, 4 * C)  # (N, H/2*W/2, 4C)
        return self.reduction(self.norm(x))                            # (N, H/2*W/2, 2C)

#### Step 6: Full Swin Transformer

Assemble everything: patch partition → linear embedding → Swin stages with patch merging → global average pool → classifier.

For our small MNIST model (32×32, 1 channel):
- **Stage 1**: 8×8 tokens, embed_dim=32, 2 blocks (W-MSA + SW-MSA)
- **Patch Merging**: 8×8 → 4×4, 32 → 64
- **Stage 2**: 4×4 tokens, 64 dims, 2 blocks
- **Head**: global average pool → Linear(64, 10)

In [ ]:
class SwinTransformer(nn.Module):

    def __init__(
        self,
        img_size=32,
        patch_size=4,
        in_chans=1,
        num_classes=10,
        embed_dim=32,
        depths=(2, 2),
        num_heads=(2, 4),
        window_size=4,
        mlp_ratio=4.0,
    ):
        super().__init__()
        self.patch_size  = patch_size
        self.window_size = window_size

        # Patch partition + linear embedding
        patch_dim = in_chans * patch_size * patch_size          # 1*4*4 = 16
        self.linear_embedding = nn.Linear(patch_dim, embed_dim)
        self.norm_embed       = nn.LayerNorm(embed_dim)

        # Build stages
        self.stages     = nn.ModuleList()
        self.downsample = nn.ModuleList()

        resolution = img_size // patch_size   # starts at 8
        dim        = embed_dim

        for i, depth in enumerate(depths):
            blocks = nn.ModuleList([
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=(resolution, resolution),
                    num_heads=num_heads[i],
                    window_size=window_size,
                    shift_size=0 if j % 2 == 0 else window_size // 2,
                    mlp_ratio=mlp_ratio,
                )
                for j in range(depth)
            ])
            self.stages.append(blocks)

            if i < len(depths) - 1:
                self.downsample.append(PatchMerging((resolution, resolution), dim))
                resolution //= 2
                dim        *= 2

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = patch_partition(x, self.patch_size)          # (N, 64, 16)
        x = self.norm_embed(self.linear_embedding(x))    # (N, 64, C)

        for i, blocks in enumerate(self.stages):
            for block in blocks:
                x = block(x)
            if i < len(self.downsample):
                x = self.downsample[i](x)

        x = self.norm(x).mean(dim=1)   # global average pool
        return self.head(x)


model = SwinTransformer(
    img_size=32, patch_size=4, in_chans=1, num_classes=10,
    embed_dim=32, depths=(2, 2), num_heads=(2, 4), window_size=4,
)
model = model.to(device)
print(model)

In [ ]:
# Sanity check: (N, 1, 32, 32) → (N, 10)
with torch.no_grad():
    out = model(torch.zeros(2, 1, 32, 32).to(device))
    print('Output shape:', out.shape)  # (2, 10)

## Training

In [ ]:
import torch.optim as optim

num_epochs = 5
lr = 1e-3

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = CrossEntropyLoss()

In [ ]:
def train(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        loss  = criterion(y_hat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (y_hat.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss / len(train_loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in tqdm(loader):
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            total_loss += criterion(y_hat, y).item()
            correct    += (y_hat.argmax(1) == y).sum().item()
            total      += len(y)
    return total_loss / len(loader), correct / total

In [ ]:
def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    return int(elapsed // 60), int(elapsed % 60)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

best_valid_loss = float('inf')
save_path       = f'models/{model.__class__.__name__}.pt'

train_losses, valid_losses = [], []

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    epoch_mins, epoch_secs = epoch_time(start_time, time.time())

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), save_path)

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'  Val.  Loss: {valid_loss:.4f} | Val.  Acc: {valid_acc*100:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(train_losses, label='train loss')
ax.plot(valid_losses, label='valid loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%')